# Analyse exploratoire — Trends Tourisme international en Afrique

## 01 — Introduction et objectifs de l'EDA

Le projet Trends de Gaea21 étudie le tourisme international dans sept destinations : **Afrique du Sud, Égypte, Kenya, Maroc, Maurice, Tanzanie et Tunisie**.

Le dataset maître rassemble les **arrivées**, les **recettes** et la **provenance** des visiteurs. L'EDA doit identifier les données exploitables, préciser les comparaisons défendables et préparer la sélection des indicateurs du futur dashboard.

Questions qui guideront l'EDA :
- Comment les arrivées et les recettes ont-elles évolué ?
- Quelles différences observe-t-on entre destinations ?
- Quelles ruptures temporelles sont visibles ?
- Quelles provenances sont documentées ?
- Quelles comparaisons sont réellement défendables ?

**Périmètre de cette phase :** introduction, chargement, validation, structure et qualité. Les analyses touristiques et les KPI seront développés ultérieurement.

> **Règles méthodologiques**
>
> Valeur manquante ≠ zéro (*missing ≠ zero*). Aucune interpolation, fabrication de valeur ou suppression silencieuse.
> La comparabilité précède la visualisation : unités, métriques, sources et couvertures doivent être compatibles.
> Les pays, régions, totaux, diasporas et institutions restent distincts ; un Top N ou un panel reste partiel.
> Les parts restent en décimal dans les données. Les parts tanzaniennes ne sont pas converties en volumes.
> La provenance égyptienne ne permet aucun classement exhaustif des marchés.

Référentiels : [documentation du projet](../docs/project_documentation.md), [sources](../docs/data_sources.md), [méthodologie](../docs/methodology.md), [dictionnaire](../docs/data_dictionary.md) et [guide de reprise](../docs/handover_guide.md).

## 02 — Chargement et validation du dataset maître

**Objectif :** charger le CSV corrigé depuis la racine du dépôt ou depuis `notebooks/`, puis contrôler son contrat de données. Les fonctions de `src/data_processing.py` sont réutilisées ; la procédure de correction n'est pas exécutée dans l'EDA.

In [1]:
import sys
from pathlib import Path
import hashlib

# Évite de créer des caches dans src lors de cette lecture seule.
sys.dont_write_bytecode = True
import pandas as pd
from IPython.display import display, Markdown

RELATIVE_DATA_PATH = Path("data/final/dataset_maitre_trends_tourisme_afrique.csv")


def find_project_root(start):
    """Cherche le fichier maître depuis le répertoire courant ou son parent."""
    start = Path(start).resolve()
    for candidate in (start, start.parent):
        if (candidate / RELATIVE_DATA_PATH).is_file() and (candidate / "src/data_processing.py").is_file():
            return candidate
    raise FileNotFoundError(
        f"Dataset maître ou module de chargement introuvable depuis {start}. "
        "Ouvrir le notebook depuis la racine du dépôt ou le dossier notebooks."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import (
    load_master_dataset,
    get_dataset_overview,
    get_observations_by_destination,
    get_value_quality_by_layer,
    get_temporal_coverage,
)

In [2]:
DATA_PATH = PROJECT_ROOT / RELATIVE_DATA_PATH
assert DATA_PATH.is_file(), f"Fichier absent : {DATA_PATH}"
source_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

df = load_master_dataset(DATA_PATH)
df_initial = df.copy(deep=True)  # Témoin d'intégrité, sans transformation.
df.head()

,dataset_layer,destination,iso3,year,origin_name,origin_region,granularity,metric,value,unit,metric_type,coverage_scope,source_name,source_reference,source_file,quality_flag,notes
0,arrivals,Afrique du Sud,ZAF,1995,NaN,NaN,destination_total,tourist_arrivals,4684000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
1,arrivals,Afrique du Sud,ZAF,1996,NaN,NaN,destination_total,tourist_arrivals,5186000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
2,arrivals,Afrique du Sud,ZAF,1997,NaN,NaN,destination_total,tourist_arrivals,5170000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
3,arrivals,Afrique du Sud,ZAF,1998,NaN,NaN,destination_total,tourist_arrivals,5898000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
4,arrivals,Afrique du Sud,ZAF,1999,NaN,NaN,destination_total,tourist_arrivals,6026000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.


**Lecture de l'aperçu.** Les premières lignes appartiennent aux arrivées nationales : les champs d'origine y sont sans objet. Cet aperçu ne représente pas toute la diversité des trois couches.

**Validation du schéma.** Le contrat actuel comporte 17 colonnes dans un ordre défini et 1 080 lignes. Toute divergence bloque la suite ; aucune correction automatique n'est appliquée.

In [3]:
EXPECTED_COLUMNS = [
    "dataset_layer", "destination", "iso3", "year", "origin_name",
    "origin_region", "granularity", "metric", "value", "unit",
    "metric_type", "coverage_scope", "source_name", "source_reference",
    "source_file", "quality_flag", "notes",
]


def control_table(checks):
    """Présente les contrôles avant de bloquer sur une éventuelle anomalie."""
    table = pd.DataFrame(checks, columns=["Contrôle", "Résultat attendu", "Résultat observé", "valide"])
    table["Statut"] = table.pop("valide").map({True: "OK", False: "ÉCHEC"})
    display(table)
    assert table["Statut"].eq("OK").all(), "Contrôles échoués : consulter le tableau, sans modifier les données."
    return table


missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
schema_controls = control_table([
    ("Nombre de colonnes", "17", str(df.shape[1]), df.shape[1] == 17),
    ("Colonnes manquantes", "Aucune", ", ".join(missing_columns) or "Aucune", not missing_columns),
    ("Colonnes supplémentaires", "Aucune", ", ".join(extra_columns) or "Aucune", not extra_columns),
    ("Ordre des colonnes", "Ordre du dictionnaire", "Conforme" if list(df.columns) == EXPECTED_COLUMNS else "Différent", list(df.columns) == EXPECTED_COLUMNS),
    ("Nombre de lignes", "1080", str(len(df)), len(df) == 1080),
])

,Contrôle,Résultat attendu,Résultat observé,Statut
0,Nombre de colonnes,17,17,OK
1,Colonnes manquantes,Aucune,Aucune,OK
2,Colonnes supplémentaires,Aucune,Aucune,OK
3,Ordre des colonnes,Ordre du dictionnaire,Conforme,OK
4,Nombre de lignes,1080,1080,OK


**Interprétation.** Le schéma correspond à la version corrigée attendue. Cette conformité structurelle ne certifie ni l'exactitude des publications originales ni la comparabilité de toutes les lignes.

**Validation des catégories.** Les modalités ci-dessous reprennent le dictionnaire, y compris `institutional_category` et `missing_unverified`. Les valeurs inattendues, les modalités absentes et les champs catégoriels vides sont signalés.

In [4]:
DESTINATION_ISO = {
    "Afrique du Sud": "ZAF", "Égypte": "EGY", "Kenya": "KEN",
    "Maroc": "MAR", "Maurice": "MUS", "Tanzanie": "TZA", "Tunisie": "TUN",
}
DESTINATIONS = list(DESTINATION_ISO)
LAYERS = ["arrivals", "receipts", "provenance"]
EXPECTED_CATEGORIES = {
    "dataset_layer": set(LAYERS),
    "destination": set(DESTINATIONS),
    "iso3": set(DESTINATION_ISO.values()),
    "granularity": {"destination_total", "aggregate_total", "country", "regional_aggregate", "diaspora", "institutional_category"},
    "metric": {"tourist_arrivals", "tourism_receipts", "tourist_origin"},
    "unit": {"persons", "current_USD", "share"},
    "metric_type": {"destination_total", "tourist_arrivals", "diaspora_arrivals", "source_market_share", "regional_tourist_share", "regional_tourist_nights_share"},
    "quality_flag": {"available", "missing_in_source", "missing_unverified", "exact_aggregate", "exact_country", "exact_diaspora", "exact_top30", "survey_share_top15", "exact_main7", "exact_panel18", "exact_single_country", "regional_share_only"},
}
category_rows = []
for column, expected in EXPECTED_CATEGORIES.items():
    observed = set(df[column].dropna().unique())
    unexpected, absent = observed - expected, expected - observed
    null_count = int(df[column].isna().sum())
    category_rows.append({
        "Variable": column,
        "Modalités documentées": ", ".join(sorted(expected)),
        "Modalités observées": ", ".join(sorted(observed)),
        "Inattendues": ", ".join(sorted(unexpected)) or "Aucune",
        "Documentées absentes": ", ".join(sorted(absent)) or "Aucune",
        "Valeurs manquantes": null_count,
        "Statut": "OK" if not unexpected and not absent and null_count == 0 else "ÉCHEC",
    })
category_controls = pd.DataFrame(category_rows)
display(category_controls)
assert category_controls["Statut"].eq("OK").all(), "Modalités différentes du dictionnaire : investigation nécessaire."
assert df["iso3"].eq(df["destination"].map(DESTINATION_ISO)).all(), "Couple destination/ISO3 incohérent."

,Variable,Modalités documentées,Modalités observées,Inattendues,Documentées absentes,Valeurs manquantes,Statut
0,dataset_layer,"arrivals, provenance, receipts","arrivals, provenance, receipts",Aucune,Aucune,0,OK
1,destination,"Afrique du Sud, Kenya, Maroc, Maurice, Tanzani...","Afrique du Sud, Kenya, Maroc, Maurice, Tanzani...",Aucune,Aucune,0,OK
2,iso3,"EGY, KEN, MAR, MUS, TUN, TZA, ZAF","EGY, KEN, MAR, MUS, TUN, TZA, ZAF",Aucune,Aucune,0,OK
3,granularity,"aggregate_total, country, destination_total, d...","aggregate_total, country, destination_total, d...",Aucune,Aucune,0,OK
4,metric,"tourism_receipts, tourist_arrivals, tourist_or...","tourism_receipts, tourist_arrivals, tourist_or...",Aucune,Aucune,0,OK
5,unit,"current_USD, persons, share","current_USD, persons, share",Aucune,Aucune,0,OK
6,metric_type,"destination_total, diaspora_arrivals, regional...","destination_total, diaspora_arrivals, regional...",Aucune,Aucune,0,OK
7,quality_flag,"available, exact_aggregate, exact_country, exa...","available, exact_aggregate, exact_country, exa...",Aucune,Aucune,0,OK


**Interprétation.** Aucune modalité inattendue n'apparaît. `country` reste une catégorie du modèle : Réunion y est conservée comme marché distinct et ne doit pas être fusionnée avec France. L'institution ONU et les groupes régionaux doivent rester séparés des pays.

**Cohérences et corrections approuvées.** Contrôler les types numériques, les unités des couches nationales, les parts décimales et les trois corrections. Les champs de traçabilité doivent être renseignés ; leur présence n'est pas une vérification de source.

In [5]:
national_mask = df["dataset_layer"].isin(["arrivals", "receipts"])
provenance_mask = df["dataset_layer"].eq("provenance")
scand = df.loc[df["destination"].eq("Tunisie") & df["origin_name"].eq("Scandinaves")]
uno = df.loc[df["destination"].eq("Kenya") & df["origin_name"].eq("United Nations Organization")]
tun_missing = df.loc[provenance_mask & df["destination"].eq("Tunisie") & df["value"].isna() & df["year"].isin([2017, 2018])]
trace_columns = ["source_name", "source_reference", "coverage_scope", "notes"]
expected_metric = df["dataset_layer"].map({"arrivals": "tourist_arrivals", "receipts": "tourism_receipts", "provenance": "tourist_origin"})
national_units = df.loc[national_mask, "dataset_layer"].map({"arrivals": "persons", "receipts": "current_USD"})
share_values = df.loc[df["unit"].eq("share"), "value"].dropna()

integrity_controls = control_table([
    ("Année entière sans absence", "Vrai", str(pd.api.types.is_integer_dtype(df["year"]) and df["year"].notna().all()), pd.api.types.is_integer_dtype(df["year"]) and df["year"].notna().all()),
    ("Value numérique", "Vrai", str(pd.api.types.is_numeric_dtype(df["value"])), pd.api.types.is_numeric_dtype(df["value"])),
    ("Metric cohérente avec la couche", "Vrai", str(df["metric"].eq(expected_metric).all()), df["metric"].eq(expected_metric).all()),
    ("Unités nationales", "persons / current_USD", ", ".join(sorted(df.loc[national_mask, "unit"].unique())), df.loc[national_mask, "unit"].eq(national_units).all()),
    ("Totaux nationaux", "destination_total", ", ".join(df.loc[national_mask, "granularity"].unique()), df.loc[national_mask, ["granularity", "metric_type"]].eq("destination_total").all().all()),
    ("Parts décimales", "Entre 0 et 1", f"{share_values.min():.3f} à {share_values.max():.3f}", share_values.between(0, 1).all()),
    ("Traçabilité renseignée", "0 champ vide", str(int(df[trace_columns].isna().sum().sum())), df[trace_columns].notna().all().all() and df[trace_columns].apply(lambda s: s.str.strip().ne("").all()).all()),
    ("Scandinaves", "7 lignes, 2017–2023, agrégat", str(len(scand)), len(scand) == 7 and set(scand["year"]) == set(range(2017, 2024)) and scand["granularity"].eq("regional_aggregate").all() and scand["quality_flag"].eq("exact_aggregate").all()),
    ("ONU Kenya", "1 ligne 2022, institution, exact_top30", str(len(uno)), len(uno) == 1 and uno["year"].eq(2022).all() and uno["granularity"].eq("institutional_category").all() and uno["quality_flag"].eq("exact_top30").all()),
    ("Absences tunisiennes", "60 lignes missing_unverified", str(len(tun_missing)), len(tun_missing) == 60 and tun_missing["quality_flag"].eq("missing_unverified").all() and tun_missing["value"].isna().all()),
])

,Contrôle,Résultat attendu,Résultat observé,Statut
0,Année entière sans absence,Vrai,True,OK
1,Value numérique,Vrai,True,OK
2,Metric cohérente avec la couche,Vrai,True,OK
3,Unités nationales,persons / current_USD,"current_USD, persons",OK
4,Totaux nationaux,destination_total,destination_total,OK
5,Parts décimales,Entre 0 et 1,0.016 à 0.643,OK
6,Traçabilité renseignée,0 champ vide,0,OK
7,Scandinaves,"7 lignes, 2017–2023, agrégat",7,OK
8,ONU Kenya,"1 ligne 2022, institution, exact_top30",1,OK
9,Absences tunisiennes,60 lignes missing_unverified,60,OK


**Interprétation.** Les trois corrections approuvées sont présentes. Les parts restent en décimal et les couches nationales gardent leurs unités distinctes. Les causes des absences tunisiennes et certaines références originales restent à vérifier.

**Clés et doublons.** Les clés logiques suivent le dictionnaire : couche–destination–année–métrique–type pour les séries nationales ; couche–destination–année–origine–granularité–type–unité pour la provenance. On compte les lignes en excès avec `keep="first"` et les lignes impliquées avec `keep=False`. Aucune ligne n'est supprimée.

In [6]:
NATIONAL_KEY = ["dataset_layer", "destination", "year", "metric", "metric_type"]
PROVENANCE_KEY = ["dataset_layer", "destination", "year", "origin_name", "granularity", "metric_type", "unit"]
duplicate_checks = [
    ("Complets", df, None),
    ("Logiques — séries nationales", df.loc[national_mask], NATIONAL_KEY),
    ("Logiques — provenance", df.loc[provenance_mask], PROVENANCE_KEY),
]
duplicate_rows = []
for label, subset, key in duplicate_checks:
    excess = int(subset.duplicated(subset=key).sum())
    involved = subset.duplicated(subset=key, keep=False)
    duplicate_rows.append({
        "Contrôle": label, "Lignes en excès": excess,
        "Lignes impliquées": int(involved.sum()),
        "Clés incomplètes": int(subset[key].isna().any(axis=1).sum()) if key else 0,
    })
    if involved.any():
        display(subset.loc[involved])
duplicate_summary = pd.DataFrame(duplicate_rows)
display(duplicate_summary)
assert duplicate_summary[["Lignes en excès", "Clés incomplètes"]].eq(0).all().all(), "Doublons ou clés incomplètes : aucune suppression automatique."

,Contrôle,Lignes en excès,Lignes impliquées,Clés incomplètes
0,Complets,0,0,0
1,Logiques — séries nationales,0,0,0
2,Logiques — provenance,0,0,0


**Interprétation.** Aucun doublon complet ou logique n'est détecté et les clés sont renseignées. Des lignes de même destination et année appartenant à des couches ou origines différentes sont légitimes.

## 03 — Structure et qualité des données

Avant l'analyse statistique, distinguer les lignes du dataset, les valeurs renseignées et les périmètres réellement comparables. Les contrôles ci-dessous ne modifient aucune donnée.

### 3.1 Dimensions du dataset

**Question :** quelle est la taille du dataset et quel périmètre général représente-t-il ?

In [7]:
overview = get_dataset_overview(df)
dimensions = pd.DataFrame({
    "Mesure": ["Lignes", "Colonnes", "Destinations", "Couches", "Première année (lignes)", "Dernière année (lignes)"],
    "Résultat": [overview["shape"][0], overview["shape"][1], df["destination"].nunique(), df["dataset_layer"].nunique(), df["year"].min(), df["year"].max()],
})
display(dimensions)
display(pd.DataFrame({"Destination": DESTINATIONS, "ISO3": [DESTINATION_ISO[d] for d in DESTINATIONS]}))

,Mesure,Résultat
0,Lignes,1080
1,Colonnes,17
2,Destinations,7
3,Couches,3
4,Première année (lignes),1995
5,Dernière année (lignes),2024


,Destination,ISO3
0,Afrique du Sud,ZAF
1,Égypte,EGY
2,Kenya,KEN
3,Maroc,MAR
4,Maurice,MUS
5,Tanzanie,TZA
6,Tunisie,TUN


**Interprétation.** Le dataset réunit 1 080 lignes, 17 colonnes, sept destinations et trois couches sur 1995–2024. Cette période globale n'est pas celle de chaque série. Le nombre de lignes par destination ne mesure pas son importance touristique.

### 3.2 Types de variables

**Question :** quels types pandas et quels rôles analytiques portent les 17 variables ?

In [8]:
roles = [
    "Couche analytique", "Destination étudiée", "Code géographique de destination",
    "Année de référence", "Libellé source du marché ou groupe", "Région selon la source",
    "Niveau statistique", "Famille d'indicateur", "Valeur à lire avec son contexte",
    "Unité de mesure", "Nature précise de la mesure", "Périmètre réellement couvert",
    "Producteur ou source déclarée", "Référence documentaire", "Fichier de la chaîne de collecte",
    "Disponibilité / qualité / couverture", "Précautions et limites",
]
variable_types = pd.DataFrame({
    "variable": EXPECTED_COLUMNS,
    "dtype": [str(df[c].dtype) for c in EXPECTED_COLUMNS],
    "rôle": roles,
})
display(variable_types)

,variable,dtype,rôle
0,dataset_layer,object,Couche analytique
1,destination,object,Destination étudiée
2,iso3,object,Code géographique de destination
3,year,int64,Année de référence
4,origin_name,object,Libellé source du marché ou groupe
5,origin_region,object,Région selon la source
6,granularity,object,Niveau statistique
7,metric,object,Famille d'indicateur
8,value,float64,Valeur à lire avec son contexte
9,unit,object,Unité de mesure


**Interprétation.** `year` est entier et `value` numérique ; les autres champs portent le contexte. Les valeurs de `value` ne peuvent pas être additionnées globalement : personnes, USD courants et parts ne mesurent pas la même chose.

### 3.3 Valeurs manquantes

**Questions :** quelles colonnes sont incomplètes, et où les valeurs numériques manquent-elles ? Séparer les champs d'origine sans objet, les absences confirmées dans la source et les causes non vérifiées.

In [9]:
missing_by_column = pd.DataFrame({
    "Variable": df.columns,
    "Valeurs manquantes": df.isna().sum().to_numpy(),
    "Part des lignes (%)": (df.isna().mean() * 100).round(2).to_numpy(),
})
display(Markdown("**A — Absences par colonne**"))
display(missing_by_column)
display(Markdown("**B — Valeurs numériques par couche**"))
controle_valeurs = get_value_quality_by_layer(df).rename(columns={
    "observations": "lignes", "valeurs_disponibles": "valeurs renseignées", "valeurs_manquantes": "valeurs manquantes",
}).reindex(LAYERS)
display(controle_valeurs)
display(Markdown("**C — Valeurs numériques par destination**"))
missing_by_destination = df.groupby("destination")["value"].agg(
    lignes="size", renseignees="count", manquantes=lambda s: s.isna().sum(),
).reindex(DESTINATIONS).rename(columns={"renseignees": "valeurs renseignées", "manquantes": "valeurs manquantes"})
display(missing_by_destination)

**A — Absences par colonne**

,Variable,Valeurs manquantes,Part des lignes (%)
0,dataset_layer,0,0.0
1,destination,0,0.0
2,iso3,0,0.0
3,year,0,0.0
4,origin_name,364,33.7
5,origin_region,364,33.7
6,granularity,0,0.0
7,metric,0,0.0
8,value,67,6.2
9,unit,0,0.0


**B — Valeurs numériques par couche**

,lignes,valeurs renseignées,valeurs manquantes
dataset_layer,,,
arrivals,182,179,3
receipts,182,178,4
provenance,716,656,60


**C — Valeurs numériques par destination**

,lignes,valeurs renseignées,valeurs manquantes
destination,,,
Afrique du Sud,106,106,0
Égypte,70,69,1
Kenya,142,140,2
Maroc,169,169,0
Maurice,73,73,0
Tanzanie,97,93,4
Tunisie,423,363,60


In [10]:
display(Markdown("**D — Nature des absences et cas particuliers**"))
structural_missing = df.groupby("dataset_layer")[["origin_name", "origin_region"]].agg(lambda s: s.isna().sum()).reindex(LAYERS)
display(structural_missing.rename(columns={"origin_name": "origin_name absent", "origin_region": "origin_region absent"}))

missing_values = df.loc[df["value"].isna()]
missing_reasons = pd.DataFrame({
    "Nature": ["missing_in_source", "missing_unverified", "Autres valeurs numériques manquantes"],
    "Lignes": [
        int(missing_values["quality_flag"].eq("missing_in_source").sum()),
        int(missing_values["quality_flag"].eq("missing_unverified").sum()),
        int((~missing_values["quality_flag"].isin(["missing_in_source", "missing_unverified"])).sum()),
    ],
})
display(missing_reasons)
display(missing_values.groupby(["dataset_layer", "destination", "year", "quality_flag"], dropna=False).size().rename("valeurs manquantes").to_frame())
missing_flags = df["quality_flag"].isin(["missing_in_source", "missing_unverified"])
assert df["value"].isna().eq(missing_flags).all(), "Incohérence entre disponibilité et quality_flag."
assert df.loc[national_mask, ["origin_name", "origin_region"]].isna().all().all()
assert df.loc[provenance_mask, ["origin_name", "origin_region"]].notna().all().all()

**D — Nature des absences et cas particuliers**

,origin_name absent,origin_region absent
dataset_layer,,
arrivals,182,182
receipts,182,182
provenance,0,0


,Nature,Lignes
0,missing_in_source,7
1,missing_unverified,60
2,Autres valeurs numériques manquantes,0


valeurs manquantes
dataset_layer destination year quality_flag                          
arrivals      Kenya       2020 missing_in_source                    1
              Tanzanie    2020 missing_in_source                    1
              Égypte      2020 missing_in_source                    1
provenance    Tunisie     2017 missing_unverified                  30
                          2018 missing_unverified                  30
receipts      Kenya       2020 missing_in_source                    1
              Tanzanie    1995 missing_in_source                    1
                          1996 missing_in_source                    1
                          2020 missing_in_source                    1

**Interprétation.** Les 364 absences de chacun des champs d'origine sont structurelles et concernent les totaux nationaux. Les 67 valeurs numériques absentes se répartissent entre 7 `missing_in_source` et 60 `missing_unverified` tunisiennes en 2017–2018 ; aucune autre absence numérique n'apparaît. La cause des 60 absences tunisiennes reste inconnue : ni zéro, ni absence confirmée dans la source. Aucune interpolation n'est réalisée.

### 3.4 Doublons

**Question :** les contrôles de clés permettent-ils de poursuivre sans dédoublonnage ? Le résultat de la section 02 est réutilisé, sans recalcul ni suppression.

In [11]:
display(duplicate_summary)

,Contrôle,Lignes en excès,Lignes impliquées,Clés incomplètes
0,Complets,0,0,0
1,Logiques — séries nationales,0,0,0
2,Logiques — provenance,0,0,0


**Interprétation.** Aucun doublon n'a été détecté selon les clés retenues. Il n'y a donc aucun dédoublonnage à effectuer ; ce contrôle ne remplace pas la vérification des sources.

### 3.5 Couverture par dimension

**Question :** combien de lignes, de valeurs renseignées et de valeurs manquantes chaque destination possède-t-elle dans chaque couche ?

In [12]:
# La fonction existante compte les lignes, y compris celles sans value.
observations_par_pays = get_observations_by_destination(df).reindex(index=DESTINATIONS, columns=LAYERS)
pair_quality = df.groupby(["destination", "dataset_layer"])["value"].agg(
    lignes="size", renseignees="count", manquantes=lambda s: s.isna().sum(),
).rename(columns={"renseignees": "valeurs renseignées", "manquantes": "valeurs manquantes"})
assert observations_par_pays.equals(pair_quality["lignes"].unstack("dataset_layer").reindex(index=DESTINATIONS, columns=LAYERS))
coverage_by_dimension = pair_quality.unstack("dataset_layer").swaplevel(0, 1, axis=1)
coverage_by_dimension = coverage_by_dimension.reindex(index=DESTINATIONS, columns=pd.MultiIndex.from_product(
    [LAYERS, ["lignes", "valeurs renseignées", "valeurs manquantes"]],
    names=["dataset_layer", "mesure"],
))
display(coverage_by_dimension)

dataset_layer  arrivals                                        receipts  \
mesure           lignes valeurs renseignées valeurs manquantes   lignes   
destination                                                               
Afrique du Sud       26                  26                  0       26   
Égypte               26                  25                  1       26   
Kenya                26                  25                  1       26   
Maroc                26                  26                  0       26   
Maurice              26                  26                  0       26   
Tanzanie             26                  25                  1       26   
Tunisie              26                  26                  0       26   

dataset_layer                                         provenance  \
mesure         valeurs renseignées valeurs manquantes     lignes   
destination                                                        
Afrique du Sud                  26                  0         54   
Égypte                          26                  0         18   
Kenya                           25                  1         90   
Maroc                           26                  0        117   
Maurice                         26                  0         21   
Tanzanie                        23                  3         45   
Tunisie                         26                  0        371   

dataset_layer                                          
mesure         valeurs renseignées valeurs manquantes  
destination                                            
Afrique du Sud                  54                  0  
Égypte                          18                  0  
Kenya                           90                  0  
Maroc                          117                  0  
Maurice                         21                  0  
Tanzanie                        45                  0  
Tunisie                        311                 60

**Lecture du tableau.** Les couches nationales ont chacune 26 lignes par destination, mais certaines valeurs manquent. La provenance varie selon les périodes, les catégories et les panels publiés. Ces effectifs ne permettent ni de comparer les volumes touristiques ni de conclure à une couverture exhaustive.

**Contexte de couverture.** Identifier explicitement unités, granularités, métriques, sources et panels avant toute comparaison. Le tableau suivant inventorie des périmètres, sans sommer les valeurs touristiques.

In [13]:
context_columns = ["destination", "unit", "granularity", "metric_type", "coverage_scope", "quality_flag"]
provenance_context = df.loc[provenance_mask].groupby(context_columns, dropna=False, sort=True).agg(
    lignes=("value", "size"), renseignees=("value", "count"),
).rename(columns={"renseignees": "valeurs renseignées"}).reset_index()
display(provenance_context)
traceability_summary = df.groupby(["destination", "dataset_layer"]).agg(
    sources=("source_name", lambda s: " ; ".join(sorted(s.unique()))),
    references=("source_reference", "nunique"),
    notes_distinctes=("notes", "nunique"),
)
display(traceability_summary)

,destination,unit,granularity,metric_type,coverage_scope,quality_flag,lignes,valeurs renseignées
0,Afrique du Sud,persons,country,tourist_arrivals,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18,54,54
1,Kenya,persons,country,tourist_arrivals,Top 30 marchés sources publiés,exact_top30,89,89
2,Kenya,persons,institutional_category,tourist_arrivals,Top 30 marchés sources publiés,exact_top30,1,1
3,Maroc,persons,aggregate_total,tourist_arrivals,Série officielle incluant agrégats publiés,exact_aggregate,9,9
4,Maroc,persons,country,tourist_arrivals,Nationalités/pays publiés par Open Data Maroc,exact_country,81,81
5,Maroc,persons,diaspora,diaspora_arrivals,MRE séparés des touristes étrangers,exact_diaspora,9,9
6,Maroc,persons,regional_aggregate,tourist_arrivals,Série officielle incluant agrégats publiés,exact_aggregate,18,18
7,Maurice,persons,country,tourist_arrivals,7 principaux marchés publiés dans les annual h...,exact_main7,21,21
8,Tanzanie,share,country,source_market_share,Top 15 marchés — parts issues de l'Exit Survey,survey_share_top15,45,45
9,Tunisie,persons,country,tourist_arrivals,Nationalités publiées par ONTT,exact_country,297,297


sources  \
destination    dataset_layer                                                      
Afrique du Sud arrivals                                          World Bank WDI   
               provenance                               Statistics South Africa   
               receipts                                          World Bank WDI   
Kenya          arrivals                                          World Bank WDI   
               provenance             TRI / Directorate of Immigration Services   
               receipts                                          World Bank WDI   
Maroc          arrivals                                          World Bank WDI   
               provenance        Open Data Maroc / Ministère chargé du Tourisme   
               receipts                                          World Bank WDI   
Maurice        arrivals                                          World Bank WDI   
               provenance                                  Statistics Mauritius   
               receipts                                          World Bank WDI   
Tanzanie       arrivals                                          World Bank WDI   
               provenance     Tanzania NBS / International Visitors' Exit Su...   
               receipts                                          World Bank WDI   
Tunisie        arrivals                                          World Bank WDI   
               provenance                                                  ONTT   
               receipts                                          World Bank WDI   
Égypte         arrivals                                          World Bank WDI   
               provenance     CAPMAS Statistical Yearbook - Tourism ; User-p...   
               receipts                                          World Bank WDI   

                              references  notes_distinctes  
destination    dataset_layer                                
Afrique du Sud arrivals                1                 1  
               provenance              3                18  
               receipts                1                 1  
Kenya          arrivals                1                 1  
               provenance              3                31  
               receipts                1                 1  
Maroc          arrivals                1                 1  
               provenance              1                 1  
               receipts                1                 1  
Maurice        arrivals                1                 1  
               provenance              3                 7  
               receipts                1                 1  
Tanzanie       arrivals                1                 1  
               provenance              3                15  
               receipts                1                 1  
Tunisie        arrivals                1                 1  
               provenance              1                 3  
               receipts                1                 1  
Égypte         arrivals                1                 1  
               provenance              2                 3  
               receipts                1                 1

**Interprétation.** Kenya : Top 30 avec une catégorie institutionnelle ; Tanzanie : parts du Top 15 ; Maurice : panel de sept marchés ; Afrique du Sud : panel de 18 marchés. Maroc et Tunisie conservent des groupes et diasporas distincts. Égypte : un seul marché pays et deux métriques régionales séparées, sans classement exhaustif. Les références présentes ne prouvent pas leur vérification : la liaison CAPMAS aux dix observations États-Unis reste non vérifiable avec les pièces du dépôt. Les libellés multilingues ne sont pas harmonisés ici.

### 3.6 Couverture temporelle par destination

**Question :** les années présentes dans les lignes correspondent-elles à des valeurs renseignées ? Calculer séparément les deux couvertures, sans créer d'année ni de valeur.

In [14]:
# La fonction existante donne la couverture des lignes ; on la complète localement.
couverture_temporelle = get_temporal_coverage(df).rename(columns={
    "annee_debut": "année première ligne", "annee_fin": "année dernière ligne", "observations": "lignes",
}).set_index(["destination", "dataset_layer"])
available_df = df.loc[df["value"].notna()]
valid_coverage = available_df.groupby(["destination", "dataset_layer"]).agg(
    premiere=("year", "min"), derniere=("year", "max"),
    annees=("year", "nunique"),
).rename(columns={"premiere": "première année renseignée", "derniere": "dernière année renseignée", "annees": "années avec valeur"})
couverture_temporelle = couverture_temporelle.join(valid_coverage).join(
    pair_quality[["valeurs renseignées", "valeurs manquantes"]]
)
couverture_temporelle = couverture_temporelle.reindex(pd.MultiIndex.from_product(
    [DESTINATIONS, LAYERS], names=["destination", "dataset_layer"],
))
display(couverture_temporelle[[
    "année première ligne", "année dernière ligne", "première année renseignée",
    "dernière année renseignée", "lignes", "valeurs renseignées",
    "valeurs manquantes", "années avec valeur",
]])

année première ligne  année dernière ligne  \
destination    dataset_layer                                               
Afrique du Sud arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Égypte         arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2010                  2019   
Kenya          arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Maroc          arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2012                  2020   
Maurice        arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Tanzanie       arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Tunisie        arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2017                  2023   

                              première année renseignée  \
destination    dataset_layer                              
Afrique du Sud arrivals                            1995   
               receipts                            1995   
               provenance                          2022   
Égypte         arrivals                            1995   
               receipts                            1995   
               provenance                          2010   
Kenya          arrivals                            1995   
               receipts                            1995   
               provenance                          2022   
Maroc          arrivals                            1995   
               receipts                            1995   
               provenance                          2012   
Maurice        arrivals                            1995   
               receipts                            1995   
               provenance                          2022   
Tanzanie       arrivals                            1995   
               receipts                            1997   
               provenance                          2022   
Tunisie        arrivals                            1995   
               receipts                            1995   
               provenance                          2017   

                              dernière année renseignée  lignes  \
destination    dataset_layer                                      
Afrique du Sud arrivals                            2020      26   
               receipts                            2020      26   
               provenance                          2024      54   
Égypte         arrivals                            2019      26   
               receipts                            2020      26   
               provenance                          2019      18   
Kenya          arrivals                            2019      26   
               receipts                            2019      26   
               provenance                          2024      90   
Maroc          arrivals                            2020      26   
               receipts                            2020      26   
               provenance                          2

**Lecture de la couverture temporelle.** Les bornes des lignes surestiment parfois la disponibilité : une ligne peut exister sans valeur. Pour la provenance, une année renseignée signifie au moins une valeur, pas une couverture complète des marchés. La composition et le périmètre restent propres à chaque source.

In [15]:
def common_years(layer):
    """Intersection des années avec au moins une valeur pour chacune des 7 destinations."""
    selected = available_df.loc[available_df["dataset_layer"].eq(layer)]
    per_destination = [
        set(selected.loc[selected["destination"].eq(destination), "year"])
        for destination in DESTINATIONS
    ]
    return set.intersection(*per_destination)


common_by_layer = {layer: common_years(layer) for layer in LAYERS}
common_national = common_by_layer["arrivals"] & common_by_layer["receipts"]
last_common_national = max(common_national) if common_national else None
display(pd.DataFrame([
    {"Couche": layer, "Années communes renseignées": ", ".join(map(str, sorted(years))) or "Aucune",
     "Dernière année commune": max(years) if years else pd.NA}
    for layer, years in common_by_layer.items()
]))
post_2020 = available_df.loc[available_df["year"].gt(2020) & available_df["dataset_layer"].isin(["arrivals", "receipts"])]
display(pd.DataFrame({
    "Contrôle": ["Dernière année commune aux deux séries nationales", "Valeurs nationales après 2020", "Années communes de provenance"],
    "Résultat": [str(last_common_national) if last_common_national else "Aucune", str(len(post_2020)), str(len(common_by_layer["provenance"]))],
}))
recovery_statement = (
    "Les séries nationales ne permettent donc pas actuellement d'étudier une reprise post-Covid."
    if post_2020.empty else
    "Des valeurs nationales après 2020 existent ; leur couverture doit être évaluée avant toute étude de reprise."
)
origin_statement = (
    "Il n'existe aucune année commune renseignée de provenance aux sept destinations."
    if not common_by_layer["provenance"] else
    "Des années de provenance sont communes, sans garantie de comparabilité des périmètres."
)
display(Markdown(
    f"**Constat recalculé.** Dernière année commune aux deux séries nationales : **{last_common_national}**. "
    f"{recovery_statement} {origin_statement}"
))

,Couche,Années communes renseignées,Dernière année commune
0,arrivals,"1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002...",2019
1,receipts,"1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004...",2019
2,provenance,Aucune,<NA>


,Contrôle,Résultat
0,Dernière année commune aux deux séries nationales,2019
1,Valeurs nationales après 2020,0
2,Années communes de provenance,0


**Constat recalculé.** Dernière année commune aux deux séries nationales : **2019**. Les séries nationales ne permettent donc pas actuellement d'étudier une reprise post-Covid. Il n'existe aucune année commune renseignée de provenance aux sept destinations.

**Interprétation.** L'étendue globale jusqu'en 2024 vient de certaines provenances et ne prolonge pas les séries nationales. Une intersection d'années est une condition nécessaire, mais insuffisante, à la comparabilité. Les couvertures partielles ne doivent pas servir à reconstruire des totaux nationaux.

In [16]:
# Vérification finale : aucune mutation du DataFrame maître ni écriture du CSV.
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash, "Le fichier maître a changé."
print("Intégrité validée : données en mémoire et fichier maître inchangés.")

Intégrité validée : données en mémoire et fichier maître inchangés.


**Fin de la phase 1.** Le schéma, les catégories, les clés et la couverture ont été contrôlés. Les réserves de source et de périmètre restent applicables. Aucune analyse de tendance ni aucun KPI n'est développé dans cette phase.